In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
uploaded = files.upload()
df = pd.read_csv("ANN testing - Sheet1.csv")

print(df.head())
print(df.shape)

In [ ]:
X = df[
    [
        "Perveance (micro)",
        "Beam waist radius (mm)",
        "Convergence angle"
    ]
].values

y = df["Theta (degrees)"].values

In [ ]:
X_train = X[:12]
y_train = y[:12]

X_test = X[12:]
y_test = y[12:]

In [ ]:
# Calculate min and max from TRAINING data only

X_min = X_train.min(axis=0)
X_max = X_train.max(axis=0)

y_min = y_train.min()
y_max = y_train.max()

# Normalization function
def normalize(data, data_min, data_max):
    return 2 * (data - data_min) / (data_max - data_min) - 1

# Normalize training and testing data
X_train_norm = normalize(X_train, X_min, X_max)
X_test_norm = normalize(X_test, X_min, X_max)

y_train_norm = normalize(y_train, y_min, y_max)
y_test_norm = normalize(y_test, y_min, y_max)

print("Normalized X_train:")
print(X_train_norm)

print("\nNormalized y_train:")
print(y_train_norm)

In [ ]:
# Activation function
def tansig(x):
    return np.tanh(x) #tanh converts any input to a value between -1 and 1

# Initialize weights and biases
np.random.seed(100)

W1 = np.random.randn(3, 3) * 0.1 #Creates weights connecting 3 inputs to 3 neurons in hidden layer 1
b1 = np.zeros(3) #bias of each neuron

W2 = np.random.randn(3, 2) * 0.1 #Connecting first hiiden layer to 2nd hidden layer of neurons which contains only 2 neurons
b2 = np.zeros(2)

W3 = np.random.randn(2, 1) * 0.1 #Connects 2 hidden neurons to output neuron
b3 = np.zeros(1)

In [ ]:
def forward(X, W1, b1, W2, b2, W3, b3):

    # Hidden layer 1
    z1 = X @ W1 + b1 #performs the weights sum calculation for 3 neurons in hidden layer 1
    h1 = tansig(z1) #outputs of hidden layer 1

    # Hidden layer 2
    z2 = h1 @ W2 + b2
    h2 = tansig(z2)

    # Output layer
    z3 = h2 @ W3 + b3
    output = tansig(z3)

    return output

In [ ]:
predictions = forward(
    X_train_norm,
    W1, b1,
    W2, b2,
    W3, b3
)

print(predictions)
print("Prediction shape:", predictions.shape)

Converting all the weights and biases of the ann into a 1-D array, making it easier to compute (since LM algorithm needs to consider every trainable parameter of the ann)

In [ ]:
def pack_params(W1, b1, W2, b2, W3, b3):
    return np.concatenate([
        W1.flatten(),
        b1.flatten(),
        W2.flatten(),
        b2.flatten(),
        W3.flatten(),
        b3.flatten()
    ])

In [ ]:
def unpack_params(params):
#Unpacking is done as ann needs matrices of the correct shapes to calculate its prediction
    index = 0

    W1 = params[index:index+9].reshape(3, 3)
    index += 9

    b1 = params[index:index+3]
    index += 3

    W2 = params[index:index+6].reshape(3, 2)
    index += 6

    b2 = params[index:index+2]
    index += 2

    W3 = params[index:index+2].reshape(2, 1)
    index += 2

    b3 = params[index:index+1]

    return W1, b1, W2, b2, W3, b3

In [ ]:
def get_errors(params, X, y):

    W1, b1, W2, b2, W3, b3 = unpack_params(params)

    predictions = forward(
        X,
        W1, b1,
        W2, b2,
        W3, b3
    ).flatten()

    errors = predictions - y

    return errors

In [ ]:
params = pack_params(
    W1, b1,
    W2, b2,
    W3, b3
)

print("Number of parameters:", len(params))

In [ ]:
errors = get_errors(
    params,
    X_train_norm,
    y_train_norm
)

print(errors)
print("Number of errors:", len(errors))

In [ ]:
def calculate_jacobian(params, X, y, epsilon=1e-5):

    current_errors = get_errors(params, X, y)

    n_samples = len(current_errors)
    n_params = len(params)

    J = np.zeros((n_samples, n_params))

    for j in range(n_params):

        params_changed = params.copy()

        params_changed[j] += epsilon

        changed_errors = get_errors(
            params_changed,
            X,
            y
        )

        J[:, j] = (
            changed_errors - current_errors
        ) / epsilon

    return J

In [ ]:
J = calculate_jacobian(
    params,
    X_train_norm,
    y_train_norm
)

print("Jacobian shape:", J.shape)

In [ ]:
errors = get_errors(
    params,
    X_train_norm,
    y_train_norm
)

mse = np.mean(errors ** 2)

print("Initial MSE:", mse)

In [ ]:
mu = 0.01

In [ ]:
J = calculate_jacobian(
    params,
    X_train_norm,
    y_train_norm
)

errors = get_errors(
    params,
    X_train_norm,
    y_train_norm
)

I = np.eye(len(params))

A = J.T @ J + mu * I
g = J.T @ errors

delta = -np.linalg.solve(A, g)

In [ ]:
new_params = params + delta

In [ ]:
old_errors = get_errors(
    params,
    X_train_norm,
    y_train_norm
)

new_errors = get_errors(
    new_params,
    X_train_norm,
    y_train_norm
)

old_mse = np.mean(old_errors ** 2)
new_mse = np.mean(new_errors ** 2)

print("Old MSE:", old_mse)
print("New MSE:", new_mse)

In [ ]:
if new_mse < old_mse:

    params = new_params

    mu = mu / 10

    print("Update accepted")
    print("mu decreased")

else:

    mu = mu * 10

    print("Update rejected")
    print("mu increased")

In [ ]:
params = pack_params(
    W1, b1,
    W2, b2,
    W3, b3
)

mu = 0.01
max_epochs = 500

mse_history = []

for epoch in range(max_epochs):

    errors = get_errors(
        params,
        X_train_norm,
        y_train_norm
    )

    old_mse = np.mean(errors ** 2)

    J = calculate_jacobian(
        params,
        X_train_norm,
        y_train_norm
    )

    I = np.eye(len(params))

    A = J.T @ J + mu * I
    g = J.T @ errors

    try:
        delta = -np.linalg.solve(A, g)
    except np.linalg.LinAlgError:
        mu *= 10
        continue

    new_params = params + delta

    new_errors = get_errors(
        new_params,
        X_train_norm,
        y_train_norm
    )

    new_mse = np.mean(new_errors ** 2)

    if new_mse < old_mse:

        params = new_params

        mu = max(mu / 10, 1e-12)

        mse_history.append(new_mse)

    else:

        mu = min(mu * 10, 1e12)

        mse_history.append(old_mse)

    if epoch % 20 == 0:
        print(
            f"Epoch {epoch:4d} | "
            f"MSE = {mse_history[-1]:.8f} | "
            f"mu = {mu:.3e}"
        )

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(mse_history)

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("ANN Training using Levenberg-Marquardt")

plt.yscale("log")

plt.show()

In [ ]:
W1, b1, W2, b2, W3, b3 = unpack_params(params)

# Take new inputs from the user
P_new = float(input("Enter Perveance (micro): "))
rw_new = float(input("Enter Beam Waist Radius (mm): "))
C_new = float(input("Enter Convergence: "))

new_input = np.array([[P_new, rw_new, C_new]])

new_input_norm = normalize(
    new_input,
    X_min,
    X_max
)

theta_norm = forward(
    new_input_norm,
    W1, b1,
    W2, b2,
    W3, b3
)

theta_predicted = (
    (theta_norm + 1) / 2
    * (y_max - y_min)
    + y_min
)

print(
    "Predicted Half-Beam Cone Angle:",
    theta_predicted[0, 0],
    "degrees"
)